# PerturbRadius quickstart

This single tutorial creates a small synthetic spatial CRISPR screen with two guides, two mice and matched NTC source regions. It then calls source components, constructs distance rings, estimates the NTC-adjusted response and applies the radius evidence gate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import perturbradius as pr

print("PerturbRadius", pr.__version__)

## 1. Create a minimal spatial screen

The simulated perturbation has a true exponential length scale of 55 µm. Source rows carry either `sgGeneA_1/2` or `NTC_1/2`. Non-source guide status is left unassigned rather than being labelled unperturbed.

In [ ]:
rng = np.random.default_rng(7)
axis = np.arange(0.0, 601.0, 10.0)
xx, yy = np.meshgrid(axis, axis)
xy = np.column_stack([xx.ravel(), yy.ravel()])
perturbation_center = np.array([150.0, 300.0])
ntc_center = np.array([450.0, 300.0])
source_radius = 32.0
frames = []

for number, section in enumerate(("S1", "S2"), start=1):
    d_gene = np.linalg.norm(xy - perturbation_center, axis=1)
    d_ntc = np.linalg.norm(xy - ntc_center, axis=1)
    gene_source = d_gene <= source_radius
    ntc_source = d_ntc <= source_radius
    boundary_distance = np.maximum(d_gene - source_radius, 0.0)
    response = 0.8 * np.exp(-boundary_distance / 55.0)
    response += rng.normal(0.0, 0.012, len(response))

    target = np.full(len(response), "unassigned", dtype=object)
    guide = np.full(len(response), "", dtype=object)
    target[gene_source] = "GeneA"
    guide[gene_source] = f"sgGeneA_{number}"
    target[ntc_source] = "NTC"
    guide[ntc_source] = f"NTC_{number}"

    frames.append(pd.DataFrame({
        "section": section,
        "mouse": f"M{number}",
        "x_um": xy[:, 0],
        "y_um": xy[:, 1],
        "target_gene": target,
        "guide": guide,
        "is_source": gene_source | ntc_source,
        "is_ntc": ntc_source,
        "response": response,
    }))

data = pd.concat(frames, ignore_index=True)
data.shape, data["is_source"].sum()

## 2. Run the complete minimal workflow

In [ ]:
result = pr.analyze_radius(
    data,
    perturbation="GeneA",
    response_col="response",
    ring_edges=(0, 20, 40, 80, 120, 180),
    eps_um=15,
    min_samples=4,
    min_source_bins=15,
    bin_size_um=10,
    replicate_col="mouse",
)

result.components.summary

## 3. Inspect the empirical curve and evidence decision

The fitted length scale is diagnostic. A biological radius is reportable only when `radius_decision.reportable` is true.

In [ ]:
display(result.effect_table[[
    "ring", "distance_midpoint_um", "effect",
    "n_perturbation_components", "n_ntc_components"
]])
print(result.model_fit)
print(result.radius_decision)

In [ ]:
ax = pr.plot_distance_response(result)
ax.set_title("Synthetic GeneA distance response")
plt.show()

## Applying this to real data

Rename your metadata columns to the schema shown in the README, set `is_source=True` only for guide-confident source cells or bins, and pass a true animal identifier through `replicate_col`. Do not treat guide non-detection as proof of an unperturbed target, and do not interpret DBSCAN components as independent biological clones without additional evidence.